In [18]:
import pandas as pd

In [19]:
#import datasets

#import course catalog dataset
df = pd.read_csv('../../datasets/2025-sp.csv') #df for data frame

#import course GPA dataset
df_GPA = pd.read_csv('../../datasets/uiuc-gpa-dataset.csv')

#import teacher rating dataset
df_Rate = pd.read_csv('../../datasets/uiuc-tre-dataset.csv')

#import subject --> unit dataset
df_Code = pd.read_csv('../../datasets/ALL-subject-code.csv')

In [20]:
#modify name function modify_Name(str), modify_Name2(str), find_code(str, mapping_df)
def modify_Name(name):
    if not isinstance(name, str) or not name.strip():
        return name  # Handle non-string or empty values
    
    name = name.strip()
    
    if ',' in name:
        # Format: "Last, First"
        parts = name.split(',')
        last_name = parts[0].strip()
        first_initial = parts[1].strip()[:1].upper() if len(parts) > 1 else ''
    else:
        # Format: "First Last"
        parts = name.split()
        if len(parts) == 0:
            return name
        elif len(parts) == 1:
            last_name = parts[0]
            first_initial = ''
        else:
            first_initial = parts[0][:1].upper()
            last_name = parts[-1]
    
    if first_initial:
        return f"{last_name}, {first_initial}"
    else:
        return last_name


def modify_Name2(name):
    if not isinstance(name, str):
        return name  # Handle non-string values
    
    # Split the name by comma
    parts = name.split(',')
    if len(parts) != 2:
        return name  # Return as-is if format is unexpected
    
    last_name = parts[0].strip()
    first_name = parts[1].strip().split()[0]  # Take only the first name
    
    # Combine in "First Last" format
    return f"{first_name} {last_name}"

def find_code(subjectt, mapping_df):
    subject = subjectt.lower()
    for idx, row in mapping_df.iterrows():
        if subject in row['variants']:
            return row['Code']
    return "N/A"


In [21]:
#df cleaning and modification (main)

#delete unnecesary columns from main dataframe
drop_cols = ['Term', 'Section Info','Schedule Information', 'Section Title', 'Enrollment Status','Status Code', 'Section Status', 'Section Credit Hours']
df.drop(drop_cols, axis=1, inplace=True) 
df['Primary Instructor (Concat)'] = df['Instructors'].apply(lambda x: str(x).split(';')[0]) #add column for primary instructor
df.rename(columns={'Type Code': 'Sched Type', 'Subject': 'Code'}, inplace=True) #rename columns
df['Number'] = df['Number'].astype(str)

In [22]:
#df_GPA cleaning and modification

#only consider data from past 10 years (20 total bc 2 semesters per year)
df_GPA = df_GPA[df_GPA['Year'] >= 2016] #only consider data from past 10 years (20 total bc 2 semesters per year)

#more renaming and typecasting
df_GPA.rename(columns={'Subject': 'Code'}, inplace=True)
df_GPA['Number'] = df_GPA['Number'].astype(str)

# Chaging the last name to only first initial (to match main DF)
df_GPA['Primary Instructor (Concat)'] = df_GPA['Primary Instructor'].apply(lambda name: modify_Name(name))
df_GPA['Primary Instructor'] = df_GPA['Primary Instructor'].apply(lambda name: modify_Name2(name))

# create mean_df_profBased_GPA + mean_df_classBased_GPA; mean GPA by professor for each class / only by class
group_cols_wProf = ['Code', 'Number', 'Sched Type', 'Primary Instructor', 'Primary Instructor (Concat)']
group_cols = ['Code', 'Number', 'Sched Type']
merge_cols = ['A+', 'A', 'A-', 'B+', 'B', 'B-', 'C+','C', 'C-', 'D+', 'D', 'D-', 'F', 'W', 'Students']
# by prof + class
mean_df_profBased_GPA = df_GPA.groupby(group_cols_wProf, as_index = False)[merge_cols].mean().round(2)
mean_df_profBased_GPA['Mean Grade By Professor (A+..F,W,Students)'] = mean_df_profBased_GPA[merge_cols].values.tolist()
mean_df_profBased_GPA.drop(merge_cols, axis=1, inplace=True) 
# by only class
mean_df_classBased_GPA = df_GPA.groupby(group_cols, as_index = False)[merge_cols].mean().round(2)
mean_df_classBased_GPA['Mean Grade By Class (A+..F,W,Students)'] = mean_df_classBased_GPA[merge_cols].values.tolist()
mean_df_classBased_GPA.drop(merge_cols, axis=1, inplace=True)

In [23]:
#df_Rate cleaning and modification

#renaming columns to match everything else
df_Rate.rename(columns={'unit': 'Subject', 'course': 'Number'}, inplace=True)

#only consider data from past 10 years, drop role, term columns
df_Rate['Year'] = df_Rate['term'].str.extract(r'(\d{4})').astype(float) #extracts the year from term
df_Rate = df_Rate[df_Rate['Year'] >= 2016] #only consider data from past 10 years (20 total bc 2 semesters per year)
#df_Rate.drop(columns=['Year', 'term', "role"], inplace=True)

#seperate Excellent and Outstanding ratings into binary columns 
#DF TEAM SHOULD FIGURE OUT HOW EXCELLENT AND OUTSTANDING ARE RATED...ok so we don't really know
df_Rate['Excellent'] = df_Rate['ranking'].apply(lambda x: 1 if str(x) == 'Excellent' else 0)
df_Rate['Outstanding'] = df_Rate['ranking'].apply(lambda x: 1 if str(x) == 'Outstanding' else 0)

#merge lname and fname columns to match main df primary instructor concat column
df_Rate['Primary Instructor (Concat)'] = df_Rate['lname'].str.title().str.strip() + ', ' + df_Rate['fname'].str.title().str.strip()

#aggregate(combine) all excellent and outstanding with same Subject, course number, and primary instructor (concat)
df_Rate = (df_Rate.groupby(['Subject', 'Number', 'Primary Instructor (Concat)'], as_index=True)[['Excellent', 'Outstanding']].sum()).reset_index()

df_Rate['Code'] = df_Rate['Subject'].apply(lambda x: find_code(x, df_Code))
df_Rate['Number'] = df_Rate['Number'].astype(str)
df_Rate.drop(columns=['Subject'], inplace=True)


In [24]:
#df merge w/ df_GPA + df_Rate
group_cols_wProf = ['Code', 'Number', 'Sched Type', 'Primary Instructor (Concat)'] # get rid of full name one bc OG df doesnt have that column
df = df.merge(mean_df_profBased_GPA, how='left', on=group_cols_wProf)
df = df.merge(mean_df_classBased_GPA, how='left', on=group_cols)

df = df.merge(df_Rate, how='left', on=['Number', 'Code','Primary Instructor (Concat)']) 

#save df before RMP!
df.to_csv('../../datasets/2025-sp-modified.csv', index=False)
df.columns

Index(['Year', 'YearTerm', 'Code', 'Number', 'Name', 'Description',
       'Credit Hours', 'Degree Attributes', 'CRN', 'Section', 'Part of Term',
       'Type', 'Sched Type', 'Start Time', 'End Time', 'Days of Week', 'Room',
       'Building', 'Instructors', 'Primary Instructor (Concat)',
       'Primary Instructor', 'Mean Grade By Professor (A+..F,W,Students)',
       'Mean Grade By Class (A+..F,W,Students)', 'Excellent', 'Outstanding'],
      dtype='object')

In [ ]:
# @title
#RMP integration functions. full name search: getRMP(str, coursename)

import requests
import re
import json

headers = {
    'User-Agent': 'Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36'
}

def getRMPexact(professor_full_name: str) -> float:

  try:
    response = requests.get("https://www.ratemyprofessors.com/search/professors/1112?q=" + professor_full_name, headers=headers)
    response.raise_for_status()
  except requests.exceptions.RequestException as e:
    #Error making request: {e}
    return None

  match = re.search(r'window\.__RELAY_STORE__ = (.*?);', response.text)
  if not match:
    #Could not find professor data on the page.
    return None

  try:
    data = json.loads(match.group(1))
  except json.JSONDecodeError:
    #Failed to parse JSON data
    return None

  for key, value in data.items():
    if isinstance(value, dict) and value.get('__typename') == 'Teacher':
      first_name = value.get('firstName', '')
      last_name = value.get('lastName', '')

      if f"{first_name} {last_name}".lower() == professor_full_name.lower():
        avg_rating = value.get('avgRating')
        if avg_rating is not None:
          # Return the float value directly
          return avg_rating

  return None

def getRMPfuzzy(professor_full_name: str, subject_number: str) -> float:

  try:
    response = requests.get("https://www.ratemyprofessors.com/search/professors/1112?q=" + professor_full_name, headers=headers)
    response.raise_for_status()
  except requests.exceptions.RequestException as e:
    print(f"Error making request: {e}")
    return None

  match = re.search(r'window\.__RELAY_STORE__ = (.*?);', response.text)
  if not match:
    print("Could not find professor data on the page.")
    return None

  try:
    data = json.loads(match.group(1))
  except json.JSONDecodeError:
    print("Failed to parse JSON data.")
    return None

  for key, value in data.items():
    if isinstance(value, dict) and value.get('__typename') == 'Teacher':

      try:
        teacher_response = requests.get("https://www.ratemyprofessors.com/professor/" + str(value.get('legacyId')), headers=headers)
        teacher_response.raise_for_status()
      except requests.exceptions.RequestException as e:
        continue

      match = re.search(r'window\.__RELAY_STORE__ = (.*?);', teacher_response.text)
      if not match:
        continue

      try:
        data1 = json.loads(match.group(1))
      except json.JSONDecodeError:
        continue

      for key1, value1 in data1.items():

        if isinstance(value1, dict) and value1.get('__typename') == 'Course':
            course_code = value1.get('courseName')

            if course_code == subject_number:
              avg_rating = value.get('avgRating')
              if avg_rating is not None:
                # Return the float value directly
                return avg_rating

  return None

#do RMPexact first, if return None then do RMP fuzzy
def getRMP(professor_full_name: str, subject_number: str) -> float:
  RMPexact = getRMPexact(professor_full_name)
  if RMPexact is not None:
    return RMPexact
  else:
    return getRMPfuzzy(professor_full_name, subject_number)

In [26]:
#if you want to save the modified dataframe to a csv file, uncomment the following line (CSV already uploaded in google drive!)
#df.to_csv('modified-2025-sp.csv', index=False)

DO NOT RUN THE FOLLOWING CODE ON YOUR LAPTOP UNLESS YOU WANT YOUR CODE TO RUN FOR AT LEAST 3 HOURS!
*if you just want the csv, it is uploaded in the google drive. 

In [ ]:
from functools import lru_cache
from tqdm import tqdm #need to type "pip3 install tqdm" in terminal if not installed
import pandas as pd

#AI code here for cache

#cached functions to avoid repeated queries
@lru_cache(maxsize=None)
def cached_getRMP(name, course):
    return getRMP(name, course)

#prepare an empty list to store results
rmp_results = []

chunk_size = 1000  # number of rows per chunk
num_chunks = (len(df) + chunk_size - 1) // chunk_size  # total number of chunks

for i in tqdm(range(num_chunks), desc="Processing RMP"):
    start_idx = i * chunk_size
    end_idx = min((i+1) * chunk_size, len(df))
    chunk = df.iloc[start_idx:end_idx]

    # apply the function to the chunk
    chunk_results = chunk.apply(
        lambda row: cached_getRMP(
            str(row.get('Primary Instructor', '')).strip()
            or str(row.get('Primary Instructor (Concat)', '')).strip(),
            str(row.get('Subject', '')).strip() + str(row.get('Number', '')).strip()
        ),
        axis=1
    )

    rmp_results.extend(chunk_results)

#assign results back to the DataFrame
df['RMP'] = rmp_results

In [ ]:
df.to_csv('modified+RMP-2025-sp.csv', index=False)